# 固定融资压力与现金优势信息检验
高利差对应H5现金优势为唯一方向假设。时点、窗口和制度/价格/月末控制预先固定，不训练，不改符号或滞后。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from scipy.stats import spearmanr
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/funding_information_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=plan['provider_uri'],region='us')
def load(key,artifact):
 spec=plan['sources'][key];p=root/spec['card'];s=json.loads((p/'summary.json').read_text());mp=p/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(s['recorder_uri']):
  rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);path=Path(rec.download_artifact(artifact));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];return rec.load_object(artifact)
funding=load('funding','analysis/state').set_index('datetime');prices=load('prices','inputs/frame');grouped=load('labels','inputs/grouped').xs('SOXX',level='instrument');assert grouped.index.equals(prices.index)
from src.data.funding_availability import reported_rate_spread_bp
original_spread=funding.spread_bp.copy()
funding['spread_bp']=[reported_rate_spread_bp(r,p) for r,p in zip(funding.sofr_percent,funding.reserve_rate_percent)]
precision_audit={'raw_distinct_known':int(original_spread.dropna().nunique()),'decimal_distinct_known':int(funding.spread_bp.dropna().nunique()),'maximum_numeric_change':float((original_spread-funding.spread_bp).abs().max()),'policy':'Decimal quote arithmetic, not outcome-dependent rounding'}
assert precision_audit['maximum_numeric_change']<1e-10
joined=funding.reindex(prices.index);frame=pd.DataFrame({'signal_date':prices.signal_date,'score':joined.spread_bp,'known':joined.known,'available_at':joined.available_at,'decision_at':pd.to_datetime(grouped['meta'].decision_at).dt.tz_localize('America/New_York'),'label_available_at':grouped['meta'].label_available_at,'cash_relative_h5':grouped['label'].cash_relative_h5},index=prices.index)
known=frame.known.eq(1);assert (frame.loc[known,'available_at']<frame.loc[known,'decision_at']).all();frame.loc[~known,'score']=np.nan
r5=grouped['feature'].soxx_ret5;r20=grouped['feature'].soxx_ret20;market=np.select([r5.ge(0)&r20.ge(0),r5.ge(0)&r20.lt(0),r5.lt(0)&r20.ge(0)],['uptrend','rebound','pullback'],default='decline')
cal=pd.DatetimeIndex(funding.index);month_last=pd.Series(cal,index=cal.to_period('M')).groupby(level=0).max();obs=pd.to_datetime(joined.observation_date);month_end=obs.eq(obs.dt.to_period('M').map(month_last))
frame['control_group']=joined.reserve_rate_is_iorb.fillna(-1).astype(int).astype(str)+'_'+pd.Series(market,index=frame.index)+'_'+month_end.astype(int).astype(str)
frame=frame.loc[frame.signal_date.dt.year.between(2019,2025)].copy();end=frame.signal_date.dt.year.map(lambda y:pd.Timestamp(year=y+1,month=1,day=1));mature=frame.label_available_at.notna()&frame.label_available_at.lt(end);frame.loc[~mature,'cash_relative_h5']=np.nan
assert frame.signal_date.dt.year.max()==2025
def metrics(part):
 p=part.dropna(subset=['score','cash_relative_h5'])
 if len(p)<30 or p.score.nunique()<2:return {'raw_spearman':np.nan,'controlled_rank_association':np.nan}
 x=p.groupby('control_group').score.rank(pct=True);y=p.groupby('control_group').cash_relative_h5.rank(pct=True);x=x-x.groupby(p.control_group).transform('mean');y=y-y.groupby(p.control_group).transform('mean')
 return {'raw_spearman':float(spearmanr(p.score,p.cash_relative_h5).statistic),'controlled_rank_association':float(x.corr(y))}
annual=[{'year':int(year),'calendar_rows':len(p),'known_mature_rows':int(p[['score','cash_relative_h5']].notna().all(axis=1).sum()),**metrics(p)} for year,p in frame.groupby(frame.signal_date.dt.year)];intervals=[]
for name,start,endyear in [('earlier',2019,2022),('later_reused',2023,2025)]:
 part=frame.loc[frame.signal_date.dt.year.between(start,endyear)];point=metrics(part);rng=np.random.default_rng(plan['seed']);n=len(part);draws=[]
 for _ in range(plan['resamples']):
  indexes=np.empty(n,dtype=int);indexes[0]=rng.integers(n)
  for j in range(1,n):indexes[j]=rng.integers(n) if rng.random()<1/plan['mean_block'] else (indexes[j-1]+1)%n
  draws.append(metrics(part.iloc[indexes]))
 draws=pd.DataFrame(draws)
 for metric,estimate in point.items():
  lower,upper=np.nanquantile(draws[metric],[.025,.975]);intervals.append({'window':name,'metric':metric,'calendar_rows':n,'known_mature_rows':int(part[['score','cash_relative_h5']].notna().all(axis=1).sum()),'estimate':estimate,'lower':lower,'upper':upper,'valid_draws':int(draws[metric].notna().sum())})
pass_windows=all(r['lower']>0 for r in intervals if r['metric']=='controlled_rank_association');annual_fraction=np.mean([r['controlled_rank_association']>0 for r in annual]);passed=pass_windows and annual_fraction>=.7
summary={'experiment':card.name,'decision':'information_pass_needs_independent_incremental_validation' if passed else 'not_promoted_fixed_funding_information_failed','primary_gate_pass':bool(passed),'positive_year_fraction':float(annual_fraction),'new_fits':0,'new_backtests':0,'parameter_searches':0,'objective_achieved':False,'provider_uri':plan['provider_uri'],'region':'us','as_of':'2026-10-13','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Fixed expected-positive funding spread versus gross H5 cash advantage with coarse policy/price/month-end controls. All historical windows were reused; no full baseline incremental or independent-forward claim. No same-morning availability substitution.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/frame']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_funding_information_no_fit',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,expected_sign='positive',horizon=5)
 rec.save_objects(artifact_path='inputs',plan=plan,precision_audit=precision_audit)
 rec.save_objects(artifact_path='analysis',frame=frame,annual=pd.DataFrame(annual),intervals=pd.DataFrame(intervals),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(annual).to_csv(card/'annual_information.csv',index=False);pd.DataFrame(intervals).to_csv(card/'information_intervals.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':plan['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));print(pd.DataFrame(annual).to_string(index=False));print(pd.DataFrame(intervals).to_string(index=False))


# 固定融资压力H5现金优势假设未通过

本轮0训练、0回测、0参数搜索。仅检验“较高SOFR减准备金付息率对应较高H5现金相对SOXX毛优势”，固定此前保守公布时钟，不使用有效日回填。按IOER/IORB制度、此前SOXX5/20日方向及利率观察是否落在股票月末日分组，组内百分位排名中心化；这是粗控制，不替代完整基线增量模型。

2019–2022共1008日，923行已知且标签在年界内成熟：控制后关联-0.01835，95%区块区间[-0.11780,0.10698]。2023–2025共752日，734行有效：关联0.00788，区间[-0.11065,0.09273]。各1000次、平均63日区块，保留未知与不成熟日的原日历位置。

年度控制后关联2019至2025依次0.1681、0.1137、-0.0071、-0.0782、-0.1067、0.1124、0.0260，仅4/7年为正，未达到既定70%方向门。两个窗口的正区间下界门也都失败。不能只保留2019/2020危机年份、只看原始相关，或改成更早的发布时钟来挽救结果。全部历史已被项目使用，未校正所有历史研究选择，不是独立确认性证据。

精度修复：首轮直接对float减法结果排名，将实际50种十进制基点值拆成113种微小不同值，最大差仅7.8e-14bp。首轮统计已作废并记录于superseded_attempt.json。现在使用原始十进制报价差计算，保留真实并列值，也支持真实半基点而不强制整数网格。修复依据报价单位，与收益无关；方向、期限、分组和门槛未变。2项时点/政策/报价合同测试通过，最终notebook重新执行。

决定：not_promoted_fixed_funding_information_failed。关闭此保守时点下的正向H5现金优势解释，不翻号、不缩短滞后、不改期限，不用“能描述融资压力”替代“能预测可交易净优势”。这不证明所有融资信息在所有任务上无效，也不提供对SOXL实际融资费用的估计。

当前仅是毛标签信息检验，没有产生10万美元Tiered完整策略净收益、回撤或前向晋升证据。现役策略未改，稳定净超额仍未实现。

[全部年度](annual_information.csv) · [固定窗口区间](information_intervals.csv)

Recorder `b294ecf8630b4f1382a98814000ecd6f`。
